# Part 16: Tool Calling, Multi-modal & Code Generation

> Extending LLMs with tools (function calling), generating images with DALL-E, text-to-speech, and code generation/optimization.

---


## 16.1 Tool Calling — How It Works

Tools (function calling) let LLMs invoke external functions. The loop is:
1. LLM decides which tool to call + arguments
2. Your code executes the tool
3. Result is returned to LLM
4. LLM produces final answer


In [ ]:
import json
from openai import OpenAI

client = OpenAI()

# Step 1: Define tools as JSON schemas
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["city"]
            }
        }
    }
]

# Step 2: Actual tool implementation
def get_weather(city: str, unit: str = "celsius") -> dict:
    # In production, call a real weather API
    return {"city": city, "temperature": 22, "unit": unit, "condition": "sunny"}

# Step 3: Tool execution loop
def run_with_tools(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]
    
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        # No tool call — final answer
        if not message.tool_calls:
            return message.content
        
        # Execute all tool calls
        messages.append(message)
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            
            if fn_name == "get_weather":
                result = get_weather(**fn_args)
            else:
                result = {"error": f"Unknown tool: {fn_name}"}
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

# print(run_with_tools("What's the weather like in Paris?"))


## 16.2 Multiple Tools & Tool Routing

In [ ]:
import datetime

tools_extended = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a mathematical calculation",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for current information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]

def dispatch_tool(tool_name: str, args: dict) -> str:
    """Route tool calls to their implementations."""
    if tool_name == "get_current_time":
        return datetime.datetime.now().isoformat()
    elif tool_name == "calculate":
        try:
            return str(eval(args["expression"]))  # Use ast.literal_eval in production
        except Exception as e:
            return f"Error: {e}"
    elif tool_name == "search_web":
        return f"[Mock search results for: {args['query']}]"
    return f"Unknown tool: {tool_name}"


## 16.3 Image Generation with DALL-E 3

In [ ]:
def generate_image(prompt: str, size: str = "1024x1024", quality: str = "standard") -> str:
    """Generate an image using DALL-E 3. Returns the image URL."""
    response = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size=size,           # "1024x1024", "1792x1024", "1024x1792"
        quality=quality,     # "standard" or "hd"
        n=1
    )
    return response.data[0].url

# Display in Jupyter
from IPython.display import Image as IPImage, display

# url = generate_image("A futuristic city with flying cars at sunset, digital art style")
# display(IPImage(url=url))


## 16.4 Text-to-Speech (TTS)

In [ ]:
def text_to_speech(text: str, output_path: str = "output.mp3", voice: str = "alloy") -> str:
    """Convert text to speech using OpenAI TTS.
    
    Voices: alloy, echo, fable, onyx, nova, shimmer
    Models: tts-1 (fast), tts-1-hd (quality)
    """
    response = client.audio.speech.create(
        model="tts-1",
        voice=voice,
        input=text
    )
    response.stream_to_file(output_path)
    return output_path

# Play in Jupyter
from IPython.display import Audio

# path = text_to_speech("Hello! Welcome to the Generative AI course.")
# Audio(path)


## 16.5 Full Multi-modal Airline Assistant (Advanced Gradio)

Combines text, images, audio in one app with tool calling.


In [ ]:
import gradio as gr

AIRLINE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_flight_status",
            "description": "Get the status of a flight",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string"},
                    "date": {"type": "string", "description": "YYYY-MM-DD format"}
                },
                "required": ["flight_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_seat",
            "description": "Book a seat on a flight",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string"},
                    "seat_class": {"type": "string", "enum": ["economy", "business", "first"]},
                    "passenger_name": {"type": "string"}
                },
                "required": ["flight_number", "seat_class", "passenger_name"]
            }
        }
    }
]

SYSTEM_PROMPT = """You are a helpful airline assistant. 
Use the provided tools to assist customers with flight information and bookings.
Be friendly, concise, and professional."""

def airline_chat(message: str, history: list) -> tuple:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_msg, asst_msg in history:
        messages.extend([
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": asst_msg}
        ])
    messages.append({"role": "user", "content": message})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=AIRLINE_TOOLS
    )
    reply = response.choices[0].message.content or "Let me look that up for you."
    
    # Generate audio response
    audio_path = None
    # audio_path = text_to_speech(reply)  # Uncomment to enable audio
    
    return reply, audio_path

# with gr.Blocks(theme=gr.themes.Soft()) as demo:
#     gr.Markdown("# ✈️ Airline AI Assistant")
#     with gr.Row():
#         chatbot = gr.Chatbot(height=400)
#         audio_out = gr.Audio(label="Voice Response", autoplay=True)
#     msg = gr.Textbox(placeholder="Ask about your flight...")
#     msg.submit(airline_chat, [msg, chatbot], [chatbot, audio_out])
# demo.launch()


## 16.6 Code Generation & Optimization (Python → C++)

In [ ]:
def optimize_python_to_cpp(python_code: str) -> str:
    """Ask LLM to rewrite Python code as optimized C++."""
    prompt = f"""You are an expert in high-performance C++ programming.
Convert the following Python code to optimized C++.

Requirements:
- Use modern C++17 features
- Maximize performance (avoid unnecessary copies, use references)
- Include all necessary headers
- Compile-ready code only — no explanations outside comments

Python Code:
```python
{python_code}
```

Provide only the C++ code:"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )
    return response.choices[0].message.content

# Example: Python max subarray sum
python_example = """
def max_subarray_sum(arr):
    max_sum = float('-inf')
    current_sum = 0
    for num in arr:
        current_sum = max(num, current_sum + num)
        max_sum = max(max_sum, current_sum)
    return max_sum
"""

# cpp_code = optimize_python_to_cpp(python_example)
# print(cpp_code)


In [ ]:
import subprocess
import os

def compile_and_run_cpp(cpp_code: str, input_data: str = "") -> tuple:
    """Compile C++ code and run it. Returns (output, error)."""
    # Write to temp file
    with open("/tmp/temp_code.cpp", "w") as f:
        f.write(cpp_code)
    
    # Compile
    compile_result = subprocess.run(
        ["g++", "-O2", "-std=c++17", "/tmp/temp_code.cpp", "-o", "/tmp/temp_binary"],
        capture_output=True, text=True
    )
    
    if compile_result.returncode != 0:
        return "", f"Compilation error:\n{compile_result.stderr}"
    
    # Run
    run_result = subprocess.run(
        ["/tmp/temp_binary"],
        input=input_data, capture_output=True, text=True, timeout=10
    )
    return run_result.stdout, run_result.stderr


## 16.7 Summary

| Feature | API / Method |
|---------|-------------|
| Tool calling | `tool_choice="auto"`, `tool_calls` loop |
| Image generation | `client.images.generate()`, DALL-E 3 |
| Text-to-speech | `client.audio.speech.create()`, TTS-1 |
| Code optimization | Prompt engineering + subprocess |
| Multi-modal UI | `gr.Blocks()` with multiple components |

---

**Next:** [Part 17 — RAG Deep Dive](Part17_RAG_Deep_Dive.ipynb)
